# 05.02.01 Logistic Regression Baseline

**Phase 05.02 — Baseline Models (Tabular Only)**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Train and evaluate a regularized binary Logistic Regression baseline on the 48 tabular features using Stratified 5-Fold Cross-Validation loaded from `data/processed/splits/cv_split_indices.json`. Implements hyperparameter search over regularization strengths ($C$) and penalty types.

**Strict Execution Policy:** This notebook is prepared for manual execution by the researcher. No cell execution outputs are pre-populated.

### Research Objectives

1. Establish linear classification baseline performance for binary charity eligibility (`ELIGIBLE` vs `INELIGIBLE`).
2. Evaluate L2-regularized logistic regression across regularisation strengths ($C \in [0.01, 0.1, 1.0, 10.0]$).
3. Compute out-of-fold metrics across 5 Stratified folds: Accuracy, Precision, Recall, Macro F1, and ROC-AUC.
4. Analyze model coefficients to identify key linear predictors and log performance to `models/trained/baselines/logistic_regression_metrics.json`.

### Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/selected_features.csv` (Primary tabular features)
- `data/processed/splits/cv_split_indices.json` (Stratified 5-Fold split definitions)

**Expected Outputs:**
- `models/trained/baselines/logistic_regression_metrics.json` (Cross-validation metrics and fold logs)
- `models/trained/baselines/logistic_regression_coefficients.csv` (Top model coefficients)

### 1. Environment Setup & Library Imports

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Libraries loaded. RANDOM_SEED={RANDOM_SEED}")


### 2. Setup Paths & Load Data and Fold Indices

In [ ]:
PROJECT_ROOT = Path("../../../..")
FEATURES_PATH = PROJECT_ROOT / "data" / "interim" / "features" / "selected_features.csv"
CV_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "cv_split_indices.json"
OUTPUT_DIR = PROJECT_ROOT / "models" / "trained" / "baselines"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_PATH.exists(), f"Missing features dataset: {FEATURES_PATH}"
assert CV_PATH.exists(), f"Missing CV split indices: {CV_PATH}"

df_features = pd.read_csv(FEATURES_PATH)
with open(CV_PATH, "r", encoding="utf-8") as f:
    cv_data = json.load(f)

feature_cols = cv_data["feature_columns"]
train_mask = df_features["split_partition"] == "train"
df_train = df_features.loc[train_mask].reset_index(drop=True)

X_train_full = df_train[feature_cols]
y_train_full = (df_train["eligibility"] == "ELIGIBLE").astype(int)

print(f"Loaded training development set: X={X_train_full.shape}, y={y_train_full.shape}")
print(f"Stratified folds loaded: {len(cv_data['folds'])} folds")


### 3. Hyperparameter Tuning and 5-Fold Stratified Cross-Validation

In [ ]:
candidate_C = [0.01, 0.1, 1.0, 10.0]
grid_results = []

print("Evaluating C candidates across 5-Fold CV...")
for c_val in candidate_C:
    fold_f1s = []
    for fold_info in cv_data["folds"]:
        t_idx = fold_info["train_indices"]
        v_idx = fold_info["val_indices"]
        
        X_tr, y_tr = X_train_full.iloc[t_idx], y_train_full.iloc[t_idx]
        X_va, y_va = X_train_full.iloc[v_idx], y_train_full.iloc[v_idx]
        
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=c_val, penalty="l2", solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED))
        ])
        pipe.fit(X_tr, y_tr)
        preds = pipe.predict(X_va)
        fold_f1s.append(f1_score(y_va, preds, average="macro"))
    
    mean_f1 = float(np.mean(fold_f1s))
    grid_results.append({"C": c_val, "mean_macro_f1": mean_f1})
    print(f"  C={c_val:<5} -> Mean Macro F1: {mean_f1:.4f}")

best_cfg = max(grid_results, key=lambda x: x["mean_macro_f1"])
best_C = best_cfg["C"]
print(f"Optimal C selected: {best_C} with Mean Macro F1: {best_cfg['mean_macro_f1']:.4f}")


### 4. Cross-Validation Evaluation with Optimal Hyperparameters

In [ ]:
fold_evaluations = []
oof_predictions = np.zeros(len(y_train_full))
oof_probabilities = np.zeros(len(y_train_full))

start_time = time.time()
for fold_info in cv_data["folds"]:
    f_num = fold_info["fold"]
    t_idx = fold_info["train_indices"]
    v_idx = fold_info["val_indices"]
    
    X_tr, y_tr = X_train_full.iloc[t_idx], y_train_full.iloc[t_idx]
    X_va, y_va = X_train_full.iloc[v_idx], y_train_full.iloc[v_idx]
    
    model_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=best_C, penalty="l2", solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED))
    ])
    
    model_pipeline.fit(X_tr, y_tr)
    val_preds = model_pipeline.predict(X_va)
    val_probs = model_pipeline.predict_proba(X_va)[:, 1]
    
    oof_predictions[v_idx] = val_preds
    oof_probabilities[v_idx] = val_probs
    
    f_acc = accuracy_score(y_va, val_preds)
    f_prec = precision_score(y_va, val_preds, zero_division=0)
    f_rec = recall_score(y_va, val_preds, zero_division=0)
    f_f1 = f1_score(y_va, val_preds, average="macro")
    f_auc = roc_auc_score(y_va, val_probs)
    
    fold_evaluations.append({
        "fold": f_num,
        "accuracy": float(f_acc),
        "precision": float(f_prec),
        "recall": float(f_rec),
        "macro_f1": float(f_f1),
        "roc_auc": float(f_auc)
    })
    print(f"Fold {f_num} -> Acc: {f_acc:.4f}, Prec: {f_prec:.4f}, Rec: {f_rec:.4f}, Macro-F1: {f_f1:.4f}, AUC: {f_auc:.4f}")

total_duration_ms = (time.time() - start_time) * 1000
mean_metrics = {
    "accuracy": float(np.mean([f["accuracy"] for f in fold_evaluations])),
    "precision": float(np.mean([f["precision"] for f in fold_evaluations])),
    "recall": float(np.mean([f["recall"] for f in fold_evaluations])),
    "macro_f1": float(np.mean([f["macro_f1"] for f in fold_evaluations])),
    "roc_auc": float(np.mean([f["roc_auc"] for f in fold_evaluations])),
    "f1_std": float(np.std([f["macro_f1"] for f in fold_evaluations]))
}
print(f"\nCross-Validation Summary (Logistic Regression):\n{json.dumps(mean_metrics, indent=2)}")


### 5. Export Baseline Metrics and Feature Coefficients

In [ ]:
# Fit on full training set to extract global coefficients
full_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(C=best_C, penalty="l2", solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED))
])
full_pipeline.fit(X_train_full, y_train_full)
coefficients = full_pipeline.named_steps["clf"].coef_[0]

df_coef = pd.DataFrame({"feature": feature_cols, "coefficient": coefficients})
df_coef["abs_importance"] = df_coef["coefficient"].abs()
df_coef = df_coef.sort_values(by="abs_importance", ascending=False).reset_index(drop=True)
df_coef.to_csv(OUTPUT_DIR / "logistic_regression_coefficients.csv", index=False)

# Export metrics summary JSON
metrics_payload = {
    "model_name": "Logistic Regression",
    "model_type": "baseline_linear",
    "best_hyperparameters": {"C": best_C, "penalty": "l2", "solver": "lbfgs"},
    "cv_metrics_mean": mean_metrics,
    "fold_details": fold_evaluations,
    "total_cv_time_ms": round(total_duration_ms, 2),
    "top_5_positive_coefficients": df_coef.sort_values(by="coefficient", ascending=False).head(5)[["feature", "coefficient"]].to_dict(orient="records"),
    "top_5_negative_coefficients": df_coef.sort_values(by="coefficient", ascending=True).head(5)[["feature", "coefficient"]].to_dict(orient="records")
}

with open(OUTPUT_DIR / "logistic_regression_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
print(f"Logistic Regression metrics exported successfully to: {OUTPUT_DIR / 'logistic_regression_metrics.json'}")


### Findings & Observations

- **Linear Baseline Establishment:** Evaluated L2-regularized Logistic Regression across 5 folds with StandardScaler normalization.
- **Feature Influence:** Evaluated coefficient magnitudes across socioeconomic and medical burden indicators.
- **Quarantine Adherence:** Final test set quarantined; all metrics computed strictly out-of-fold.

### Completion Checklist

- [ ] Stratified 5-Fold CV executed on training development set.
- [ ] Optimal C regularizer determined by out-of-fold Macro F1.
- [ ] Out-of-fold metrics exported to `logistic_regression_metrics.json`.
- [ ] Top coefficients saved to `logistic_regression_coefficients.csv`.
- [ ] Zero test set leakage confirmed.